# South African language identification

ExploreAI classification-hackathon project with a Codex-assisted 2026 reliability extension. This clean walkthrough reads the executed aggregate experiment reports and predicts one authored example. It does not retrain or reevaluate the holdout.


## Reproduce first

Follow the README: create the pinned environment, run the tests, prepare a fresh local run, select on validation, and evaluate the holdout once. See docs/PROTOCOL.md for the fixed comparison and limitations. Open this notebook from the project root or notebooks directory.


In [1]:
from pathlib import Path
import json
import sys
root = Path.cwd()
if not (root / "sa_lid").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
results = json.loads((root / "reports/metrics.json").read_text(encoding="utf-8"))
audit = results["audit"]
print("Raw labelled rows:", audit["raw_rows"])
print("Retained unique texts:", audit["retained_unique_texts"])
print("Duplicate rows collapsed:", audit["duplicate_rows_collapsed"])
print("Partition sizes:", audit["partition_sizes"])
print("Normalized text overlap:", audit["normalized_text_overlap"])


Raw labelled rows: 33000
Retained unique texts: 29948
Duplicate rows collapsed: 3052
Partition sizes: {'train': 17968, 'validation': 5990, 'holdout': 5990}
Normalized text overlap: 0


## Select with validation only

The two configurations were declared before training. Vectorizers were fitted only on the training rows. Character patterns are a hypothesis tested against a word-level Naive Bayes baseline; no improvement is assumed in advance.


In [2]:
for name, score in results["validation"].items():
    print(f"{name}: macro F1={score["macro_f1"]:.6f}, accuracy={score["accuracy"]:.6f}")
print("Selected:", results["selected"])
print("Holdout used for selection:", results["holdout_used_for_selection"])


word_tfidf_nb: macro F1=0.997838, accuracy=0.997830
char_tfidf_svm: macro F1=0.999357, accuracy=0.999332
Selected: char_tfidf_svm
Holdout used for selection: False


## Inspect the saved one-use holdout report

Reading an existing report does not rerun evaluation. Per-language differences are descriptive diagnostics, not a fairness certification.


In [3]:
holdout = results["holdout"]
print(f"Local holdout macro F1: {holdout["macro_f1"]:.6f}")
print("Language   precision   recall      F1          support")
for label, score in holdout["per_language"].items():
    print(f"{label:8} {score["precision"]:.6f}    {score["recall"]:.6f}    {score["f1-score"]:.6f}    {int(score["support"])}")


Local holdout macro F1: 0.999168
Language   precision   recall      F1          support
afr      1.000000    1.000000    1.000000    528
eng      0.998336    1.000000    0.999167    600
nbl      1.000000    0.995763    0.997877    472
nso      0.998264    1.000000    0.999131    575
sot      1.000000    0.998236    0.999117    567
ssw      1.000000    1.000000    1.000000    485
tsn      1.000000    1.000000    1.000000    574
tso      1.000000    1.000000    1.000000    551
ven      1.000000    1.000000    1.000000    521
xho      0.998120    0.998120    0.998120    532
zul      0.996587    0.998291    0.997438    585


## Reuse the complete fitted pipeline

This authored example is not benchmark evaluation. Use only the model generated by your trusted local run; never load historical or third-party pickle files. The classifier always chooses one of the eleven labels and has no calibrated confidence or unknown-language detector.


In [4]:
from sa_lid.workflow import predict
print(predict(root / ".local/portfolio-v1", "This example sentence is written in English."))


{'language': 'eng', 'scope': 'closed set of eleven benchmark labels; no calibrated confidence'}


## Limitations and contribution

These are local in-distribution results, not Kaggle scores or evidence of production operation. Exact duplicate control does not remove all near duplicates or shared document sources. Domain shift, dialect coverage, code-switching and unknown languages remain unverified. Original coursework and historical notebooks are retained; the new package, tests and reporting were developed with Codex assistance. An independent owner teach-back has not been assessed.
